In [1]:
import pandas as pd
import numpy as np
from utils import *

In [2]:
# df_raw = pd.read_csv('../Data/train-2026.csv')
# print(df_raw.shape)
# df_raw['default'] = df_raw.apply(lambda x: create_target_variable(x['stmt_date'], x['def_date']), axis=1)
# df = df_raw[~df_raw['default'].isna()]
# print(df.shape)

In [3]:
# df_raw.shape[0] - df.shape[0], 1 - (df.shape[0]/df_raw.shape[0])

In [4]:
# df.to_csv('../Data/train-2026_with_target_variable.csv', index=False)

In [5]:
df = pd.read_csv('../Data/train-2026_with_target_variable.csv')
df.shape

/var/folders/22/3z4yr_bd5ds4d1g7ph9nprs80000gn/T/ipykernel_88060/207617541.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../Data/train-2026_with_target_variable.csv')


(1018337, 45)

In [6]:
df.groupby(['fs_year']).agg({'id':'count','default':['sum','mean']})

id default          
          count     sum      mean
fs_year                          
2007     144128    1475  0.010234
2008     161136    2308  0.014323
2009     170125    2867  0.016852
2010     174229    2967  0.017029
2011     183730    2134  0.011615
2012     184989       0  0.000000

In [7]:
column_categorization = {
    '1. Identity_cols': ['id', 'stmt_date', 'HQ_city', 'legal_struct', 'ateco_sector', 'def_date', 'fs_year'],
    '2. Asset_cols': ['asst_intang_fixed', 'asst_tang_fixed', 'asst_fixed_fin', 'asst_current', 'AR', 'cash_and_equiv', 'asst_tot', 'goodwill'],
    '3. Equity_and_liabilities_cols': ['eqty_tot','eqty_corp_family_tot','liab_lt','liab_lt_emp','debt_bank_st','debt_bank_lt','debt_fin_st','debt_fin_lt','AP_st','AP_lt','debt_st','debt_lt'],
    '4. Income_statement_cols': ['rev_operating','COGS','prof_operations','inc_financing','exp_financing','prof_financing','inc_extraord','taxes','profit'],
    '5. Cashflow_statement_cols': ['cf_operations'],
    '6. Financial_derived_ratios': ['days_rec', 'ebitda', 'roa', 'roe', 'wc_net', 'margin_fin'],
    '7. Target': ['default']
}

assert sorted(sum(column_categorization.values(), [])) == sorted(df.drop(columns={'Unnamed: 0'}).columns)

In [8]:
missing_values = []
for category, cols in column_categorization.items():
    for c in cols:
        missing_values.append({
            'Category': category,
            'Column': c,
            'Missing': df[c].isna().sum(),
            '% Missing': df[c].isna().mean()
        })
missing_values = pd.DataFrame(missing_values)
missing_values.to_csv('../EDA/missing_values.csv')

## Some features and financial ratios (percentage match check)

#### Profit from operations

In [40]:
# prof_operations = rev_operating - COGS
select_cols = ['prof_operations', 'rev_operating', 'COGS']
df_temp = df[~df['prof_operations'].isna() & ~df['rev_operating'].isna() & ~df['COGS'].isna()][select_cols]
df_temp['prof_operations_recons'] = df_temp['rev_operating'] - df_temp['COGS']
print("Data left after dropping NaNs", np.round(df_temp.shape[0]/df.shape[0]* 100, 2))
print("Match %", (df_temp['prof_operations_recons'] == df_temp['prof_operations']).sum()/df_temp.shape[0] * 100)

Data left after dropping NaNs 99.98
Match % 99.98968730939937


#### Financial margin

In [41]:
# margin_fin = equity - (fixed assets)
select_cols = ['eqty_tot', 'asst_fixed_fin', 'asst_intang_fixed', 'asst_tang_fixed', 'margin_fin']
df_temp = df[~df['eqty_tot'].isna() & ~df['asst_fixed_fin'].isna() & ~df['asst_tang_fixed'].isna() & ~df['asst_fixed_fin'].isna() & ~df['margin_fin'].isna()]
df_temp['margin_fin_recons'] = df_temp['eqty_tot'] - (df_temp['asst_fixed_fin'] + df_temp['asst_intang_fixed'] + df_temp['asst_tang_fixed'])
print("Data left after dropping NaNs", np.round(df_temp.shape[0]/df.shape[0]* 100, 2))
print("Match %", (df_temp['margin_fin_recons'] == df_temp['margin_fin']).sum()/df_temp.shape[0] * 100)

Data left after dropping NaNs 96.25
Match % 99.83248128677225


/var/folders/22/3z4yr_bd5ds4d1g7ph9nprs80000gn/T/ipykernel_88060/475703216.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['margin_fin_recons'] = df_temp['eqty_tot'] - (df_temp['asst_fixed_fin'] + df_temp['asst_intang_fixed'] + df_temp['asst_tang_fixed'])


#### New working capital

In [42]:
# wc_net = asst_current - (debt_st + AP_st)
select_cols = ['asst_current', 'debt_st', 'wc_net', 'AP_st']
df_temp = df[~df['debt_st'].isna() & ~df['wc_net'].isna() & ~df['asst_current'].isna() & ~df['AP_st'].isna()][select_cols]
df_temp['wc_net_recons'] = df_temp['asst_current'] - (df_temp['debt_st'] + df_temp['AP_st'])
print("Data left after dropping NaNs", np.round(df_temp.shape[0]/df.shape[0]* 100, 2))
print("Match %", (df_temp['wc_net_recons'] == df_temp['wc_net']).sum()/df_temp.shape[0] * 100)

# wc_net = asst_current - debt_st
select_cols = ['asst_current', 'debt_st', 'wc_net']
df_temp = df[~df['debt_st'].isna() & ~df['wc_net'].isna() & ~df['asst_current'].isna()][select_cols]
df_temp['wc_net_recons'] = df_temp['asst_current'] - df_temp['debt_st']
print("Data left after dropping NaNs", np.round(df_temp.shape[0]/df.shape[0]* 100, 2))
print("Match %", (df_temp['wc_net_recons'] == df_temp['wc_net']).sum()/df_temp.shape[0] * 100)

Data left after dropping NaNs 88.22
Match % 3.365049611412517
Data left after dropping NaNs 100.0
Match % 99.9822258185207


#### ROA and ROE

In [43]:
print("Firms with total assets = 0: ", (df['asst_tot'] == 0).sum())

# roa = profit / asst_tot * 100
select_cols = ['profit', 'asst_tot', 'roa']
df_temp = df[~df['asst_tot'].isna() & ~df['roa'].isna() & ~df['profit'].isna()][select_cols]
df_temp['roa_recons'] = df["profit"] / df["asst_tot"] * 100
print("Data left after dropping NaNs", np.round(df_temp.shape[0]/df.shape[0]* 100, 2))
print("Match %", ((df_temp['roa_recons'] - df_temp['roa']) <= 0.01).sum()/df_temp.shape[0] * 100)

# roa = prof_operations / asst_tot * 100
select_cols = ['prof_operations', 'asst_tot', 'roa']
df_temp = df[~df['asst_tot'].isna() & ~df['roa'].isna() & ~df['prof_operations'].isna()][select_cols]
df_temp['roa_recons'] = df["prof_operations"] / df["asst_tot"] * 100
print("Data left after dropping NaNs", np.round(df_temp.shape[0]/df.shape[0]* 100, 2))
print("Match %", ((df_temp['roa_recons'] - df_temp['roa']) <= 0.01).sum()/df_temp.shape[0] * 100)

Firms with total assets = 0:  0
Data left after dropping NaNs 100.0
Match % 88.58057819384882
Data left after dropping NaNs 100.0
Match % 99.99715214423898


In [ ]:
print("Firms with total equity = 0: ", (df['eqty_tot'] == 0).sum())

# roe = profit / asst_tot * 100
select_cols = ['profit', 'asst_tot', 'roe']
df_temp = df[~df['asst_tot'].isna() & ~df['roe'].isna() & ~df['profit'].isna()][select_cols]
df_temp['roe_recons'] = df["profit"] / df["asst_tot"] * 100
print("Data left after dropping NaNs", np.round(df_temp.shape[0]/df.shape[0]* 100, 2))
print("Match %", ((df_temp['roe_recons'] - df_temp['roe']) <= 0.01).sum()/df_temp.shape[0] * 100)

# roe = prof_operations / asst_tot * 100
select_cols = ['prof_operations', 'asst_tot', 'roe']
df_temp = df[~df['asst_tot'].isna() & ~df['roe'].isna() & ~df['prof_operations'].isna()][select_cols]
df_temp['roe_recons'] = df["prof_operations"] / df["asst_tot"] * 100
print("Data left after dropping NaNs", np.round(df_temp.shape[0]/df.shape[0]* 100, 2))
print("Match %", ((df_temp['roe_recons'] - df_temp['roe']) <= 0.01).sum()/df_temp.shape[0] * 100)

Firms with total equity = 0:  14
Data left after dropping NaNs 93.17
Match % 69.99174702687434
Data left after dropping NaNs 93.17
Match % 47.91398769747077


#### Days receivables

In [45]:
# days_rec = AR / rev_operating * 365
select_cols = ['AR', 'rev_operating', 'days_rec']
df_temp = df[~df['rev_operating'].isna() & ~df['days_rec'].isna() & ~df['AR'].isna()][select_cols]
df_temp['days_rec_recons'] = df["AR"] / df["rev_operating"] * 365
print("Data left after dropping NaNs", np.round(df_temp.shape[0]/df.shape[0]* 100, 2))
print("Match %", ((df_temp['days_rec_recons'] - df_temp['days_rec']) <= 0.01).sum()/df_temp.shape[0] * 100)

Data left after dropping NaNs 27.68
Match % 26.60058677243234


#### debt st and lt

In [46]:
# debt_st = debt_fin_st + debt_bank_st
select_cols = ['debt_fin_st', 'debt_bank_st', 'debt_st']
df_temp = df[~df['debt_bank_st'].isna() & ~df['debt_st'].isna() & ~df['debt_fin_st'].isna()][select_cols]
df_temp['debt_st_recons'] = df["debt_fin_st"] + df["debt_bank_st"]
print("Data left after dropping NaNs", np.round(df_temp.shape[0]/df.shape[0]* 100, 2))
print("Match %", ((df_temp['debt_st_recons'] - df_temp['debt_st']) <= 0.01).sum()/df_temp.shape[0] * 100)

# debt_st = debt_fin_st + debt_bank_st + AP_st
select_cols = ['debt_fin_st', 'debt_bank_st', 'debt_st', 'AP_st']
df_temp = df[~df['debt_bank_st'].isna() & ~df['debt_st'].isna() & ~df['debt_fin_st'].isna() & ~df['AP_st'].isna()][select_cols]
df_temp['debt_st_recons'] = df["debt_fin_st"] + df["debt_bank_st"] + df["AP_st"]
print("Data left after dropping NaNs", np.round(df_temp.shape[0]/df.shape[0]* 100, 2))
print("Match %", ((df_temp['debt_st_recons'] - df_temp['debt_st']) <= 0.01).sum()/df_temp.shape[0] * 100)

Data left after dropping NaNs 88.01
Match % 99.7348853401412
Data left after dropping NaNs 88.01
Match % 99.42714447028158


In [47]:
# debt_lt = debt_fin_lt + debt_bank_lt
select_cols = ['debt_fin_lt', 'debt_bank_lt', 'debt_lt']
df_temp = df[~df['debt_bank_lt'].isna() & ~df['debt_lt'].isna() & ~df['debt_fin_lt'].isna()][select_cols]
df_temp['debt_lt_recons'] = df["debt_fin_lt"] + df["debt_bank_lt"]
print("Data left after dropping NaNs", np.round(df_temp.shape[0]/df.shape[0]* 100, 2))
print("Match %", ((df_temp['debt_lt_recons'] - df_temp['debt_lt']) <= 0.01).sum()/df_temp.shape[0] * 100)

# debt_lt = debt_fin_lt + debt_bank_lt + AP_lt
select_cols = ['debt_fin_lt', 'debt_bank_lt', 'debt_lt', 'AP_lt']
df_temp = df[~df['debt_bank_lt'].isna() & ~df['debt_lt'].isna() & ~df['debt_fin_lt'].isna() & ~df['AP_lt'].isna()][select_cols]
df_temp['debt_lt_recons'] = df["debt_fin_lt"] + df["debt_bank_lt"] + df["AP_lt"]
print("Data left after dropping NaNs", np.round(df_temp.shape[0]/df.shape[0]* 100, 2))
print("Match %", ((df_temp['debt_lt_recons'] - df_temp['debt_lt']) <= 0.01).sum()/df_temp.shape[0] * 100)

Data left after dropping NaNs 91.0
Match % 98.8659087352762
Data left after dropping NaNs 90.98
Match % 98.8476283399027


## Impute missing values

In [48]:
df['data_type'] = df['fs_year'].apply(
    lambda x: 'train' if x <= 2009 else 'oot1' if x == 2010 else 'oot2' if x == 2011 else '2012'
)
df.groupby(['data_type']).agg({'id':'count', 'stmt_date': ['min', 'max'], 'default': 'mean'})

id   stmt_date               default
            count         min         max      mean
data_type                                          
2012       184989  2012-12-31  2012-12-31  0.000000
oot1       174229  2010-12-31  2010-12-31  0.017029
oot2       183730  2011-12-31  2011-12-31  0.011615
train      475389  2007-12-31  2009-12-31  0.013989

In [49]:
df.query('data_type == "train"').to_csv('../Data/train.csv', index=False)
df.query('data_type == "oot1"').to_csv('../Data/oot1.csv', index=False)
df.query('data_type == "oot2"').to_csv('../Data/oot2.csv', index=False)
df.query('data_type == "2012"').to_csv('../Data/df_2012.csv', index=False)

In [82]:
train = pd.read_csv('../Data/train.csv')
oot1 = pd.read_csv('../Data/oot1.csv')
oot2 = pd.read_csv('../Data/oot2.csv')
df_2012 = pd.read_csv('../Data/df_2012.csv')

print(train.shape, oot1.shape, oot2.shape, df_2012.shape)

/var/folders/22/3z4yr_bd5ds4d1g7ph9nprs80000gn/T/ipykernel_88060/1546398685.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('../Data/train.csv')
/var/folders/22/3z4yr_bd5ds4d1g7ph9nprs80000gn/T/ipykernel_88060/1546398685.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  oot1 = pd.read_csv('../Data/oot1.csv')
/var/folders/22/3z4yr_bd5ds4d1g7ph9nprs80000gn/T/ipykernel_88060/1546398685.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  oot2 = pd.read_csv('../Data/oot2.csv')


(475389, 46) (174229, 46) (183730, 46) (184989, 46)


In [51]:
train.columns

Index(['Unnamed: 0', 'id', 'stmt_date', 'HQ_city', 'legal_struct',
       'ateco_sector', 'def_date', 'fs_year', 'asst_intang_fixed',
       'asst_tang_fixed', 'asst_fixed_fin', 'asst_current', 'AR',
       'cash_and_equiv', 'asst_tot', 'eqty_tot', 'eqty_corp_family_tot',
       'liab_lt', 'liab_lt_emp', 'debt_bank_st', 'debt_bank_lt', 'debt_fin_st',
       'debt_fin_lt', 'AP_st', 'AP_lt', 'debt_st', 'debt_lt', 'rev_operating',
       'COGS', 'prof_operations', 'goodwill', 'inc_financing', 'exp_financing',
       'prof_financing', 'inc_extraord', 'taxes', 'profit', 'days_rec',
       'ebitda', 'roa', 'roe', 'wc_net', 'margin_fin', 'cf_operations',
       'default', 'data_type'],
      dtype='object')

In [52]:
drop_cols = ["Unnamed: 0", "eqty_corp_family_tot", "days_rec", "roe", "HQ_city"]
debt_impute_cols = ["debt_bank_st", "debt_bank_lt", "debt_fin_st", "debt_fin_lt", "AP_st", "AP_lt", "debt_st", "debt_lt"]
asset_impute_cols = ['asst_intang_fixed','asst_tang_fixed','asst_fixed_fin','asst_current','AR','cash_and_equiv','goodwill']
income_statement_impute_cols = ["rev_operating", "COGS", "prof_operations", "inc_financing", "exp_financing", "prof_financing", "inc_extraord", "taxes", "profit"]

cols_to_impute = [
    *asset_impute_cols, "eqty_tot", "liab_lt", "liab_lt_emp", *debt_impute_cols, *income_statement_impute_cols, "ebitda", "cf_operations",
]

In [53]:
imputer_table = median_table_by_ateco_sector(train, cols_to_impute)
imputer_table.to_csv('../Data/imputer.csv', index=False)
imputer_df = pd.read_csv('../Data/imputer.csv')
imputer_df

,ateco_sector,asst_intang_fixed,asst_tang_fixed,asst_fixed_fin,asst_current,AR,cash_and_equiv,goodwill,eqty_tot,liab_lt,...,prof_operations,inc_financing,exp_financing,prof_financing,inc_extraord,taxes,profit,ebitda,cf_operations,asst_tot
0,1.0,0.001999,0.731236,0.000074,0.194361,0.075994,0.004522,0.029236,0.260705,0.000000,...,0.001603,-0.007504,0.008294,0.0,0.000000e+00,-0.000273,-0.001331,0.011864,0.005661,3640466.5
1,2.0,0.001476,0.462750,0.000000,0.397079,0.140536,0.007060,0.039459,0.163495,0.000000,...,0.007768,-0.006894,0.007417,0.0,0.000000e+00,0.001694,0.000226,0.018126,0.005857,2680169.0
2,3.0,0.000923,0.537728,0.000658,0.434910,0.149222,0.004823,0.091207,0.207822,0.000000,...,0.012796,-0.011804,0.012765,0.0,0.000000e+00,0.001382,0.000211,0.042697,0.023262,2867793.0
3,5.0,0.006416,0.173621,0.009986,0.770442,0.345646,0.063181,0.112824,0.314054,0.002726,...,0.041147,-0.010607,0.014264,0.0,2.754738e-03,0.020803,0.004085,0.072003,0.041589,22942135.0
4,6.0,0.052570,0.066258,0.000160,0.276109,0.154122,0.011656,-0.003559,0.398089,0.001609,...,-0.018595,0.000274,0.000295,0.0,0.000000e+00,-0.017410,-0.011603,-0.007239,-0.002495,9213390.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,94.0,0.000000,0.436739,0.000484,0.552581,0.185747,0.003095,0.068204,0.046994,0.000000,...,0.027797,-0.000299,0.011615,0.0,9.456156e-08,0.019982,0.014131,0.040366,0.024380,4338847.0
80,95.0,0.009359,0.076718,0.000688,0.833958,0.526728,0.040403,0.301541,0.169297,0.000000,...,0.045463,-0.010589,0.012432,0.0,0.000000e+00,0.028588,0.006096,0.072896,0.033789,2539506.5
81,96.0,0.006836,0.508242,0.000518,0.332433,0.225290,0.012917,0.237473,0.215606,0.000000,...,0.024102,-0.011837,0.013716,0.0,0.000000e+00,0.011720,0.001793,0.064471,0.041771,3458264.0
82,99.0,0.000082,0.000784,0.000000,0.988523,0.067743,0.001807,0.021523,0.056645,0.000000,...,0.007166,-0.002781,0.004341,0.0,0.000000e+00,0.000253,-0.000391,0.011653,0.000401,2664070.5


In [81]:
def missing_value_imputation(df, imputation_table, cols_to_impute):
    df_copy = df.copy()
    for c in df.columns:
        if c in cols_to_impute: df[c] = pd.to_numeric(df[c], errors="coerce")

    # no information
    df_copy['HQ_city'] = df_copy['HQ_city'].fillna(-1)

    # flag if debt columns are missing
    # Since we are imputing in these rows, we want our model to know that there has been imputation here
    df_copy['debt_cols_imputation_flag'] = df_copy[debt_impute_cols].isna().any(axis=1).astype(int)
    df_copy['asset_cols_imputation_flag'] = df_copy[asset_impute_cols].isna().any(axis=1).astype(int)
    df_copy['income_statement_cols_imputation_flag'] = df_copy[income_statement_impute_cols].isna().any(axis=1).astype(int)

    # reconstruct prof_operations using rev_operating - COGS
    df_copy["rev_operating"] = df_copy["rev_operating"].fillna(df_copy["prof_operations"] + df_copy["COGS"])
    df_copy["COGS"] = df_copy["COGS"].fillna(df_copy["rev_operating"] - df_copy["prof_operations"])
    df_copy["prof_operations"] = df_copy["prof_operations"].fillna(df_copy["rev_operating"] - df_copy["COGS"])

    # impute using imputation table based on ateco_sector, if absent then impute the global value
    """
        LOGIC:
            if ateco_sector and asst_tot is preset in df_copy:
                df_copy[feature] = table[ateco_sector][feature] * df_copy[asst_tot]
            elif asst_tot is not preset but ateco_sector is:
                df_copy[feature] = table[ateco_sector][feature] * table[ateco_sector][asst_tot]
            elif ateco_sector is present but asst_tot is not:
                df_copy[feature] = table[global][feature] * df_copy[asst_tot]
            else both missing:
                df_copy[feature] = table[global][feature] * table[global][asst_tot]
    """
    imputation_table = imputation_table.set_index("ateco_sector")

    row_medians = imputation_table.reindex(df_copy["ateco_sector"]).set_axis(df_copy.index)
    row_medians = row_medians.fillna(imputation_table.loc["global"])
    assets = df_copy["asst_tot"].fillna(row_medians["asst_tot"])

    df_copy[cols_to_impute] = df_copy[cols_to_impute].fillna(row_medians[cols_to_impute].mul(assets, axis=0))

    # reconstruct the columns based on the formulas
    df_copy["margin_fin"] = df_copy["margin_fin"].fillna(df_copy["eqty_tot"] - (df_copy["asst_intang_fixed"] + df_copy["asst_tang_fixed"] + df_copy["asst_fixed_fin"]))
    df_copy["wc_net"] = df_copy["wc_net"].fillna(df_copy["asst_current"] - df_copy["debt_st"])
    df_copy["roa"] = df_copy["roa"].fillna(df_copy["prof_operations"] / df_copy["asst_tot"] * 100)

    return df_copy

In [91]:
train_imputed = missing_value_imputation(train, imputer_df, cols_to_impute)
oot1_imputed = missing_value_imputation(oot1, imputer_df, cols_to_impute)
oot2_imputed = missing_value_imputation(oot2, imputer_df, cols_to_impute)
df_2012_imputed = missing_value_imputation(df_2012, imputer_df, cols_to_impute)

In [92]:
cols_to_drop = ['eqty_corp_family_tot','days_rec','roe', 'def_date']

train_imputed = train_imputed.drop(columns=cols_to_drop)
oot1_imputed = oot1_imputed.drop(columns=cols_to_drop)
oot2_imputed = oot2_imputed.drop(columns=cols_to_drop)
df_2012_imputed = df_2012_imputed.drop(columns=cols_to_drop)

In [93]:
print(train_imputed.isna().sum()[train_imputed.isna().sum() != 0])
print(oot1_imputed.isna().sum()[oot1_imputed.isna().sum() != 0])
print(oot2_imputed.isna().sum()[oot2_imputed.isna().sum() != 0])
print(df_2012_imputed.isna().sum()[df_2012_imputed.isna().sum() != 0])

Series([], dtype: int64)
Series([], dtype: int64)
Series([], dtype: int64)
Series([], dtype: int64)


In [94]:
train_imputed.to_csv('../Data/train_imputed.csv', index=False)
oot1_imputed.to_csv('../Data/oot1_imputed.csv', index=False)
oot2_imputed.to_csv('../Data/oot2_imputed.csv', index=False)
df_2012_imputed.to_csv('../Data/df_2012_imputed.csv', index=False)